# Manual validation of the predicted immune cell states

**Developed by:** Anna Maguza\
**Affiliation:** Faculty of Medicine, Würzburg University\
**Creation date:** 17th February 2024\
**Last modified date:** 17th February 2024

This notebook outlines the process of validation of predicted by `scVI - scANVI` pipeline mesenchymal cell states annotations. We will evaluate `scANVI` classificator confidence and also validate the predicted annotation with markers.

## Import packages

In [1]:
import numpy as np
import scanpy as sc
import anndata
import pandas as pd
import plotnine as p
import matplotlib.pyplot as plt
import seaborn as sns

import json
from datetime import datetime

## Setup working environment

In [65]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi = 180, color_map = 'magma_r', dpi_save = 300, vector_friendly = True, format = 'svg')

In [66]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [ ]:
timestamp

## Upload data

In [ ]:
adata = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_scVI_scANVI_immune_cellstates_AM_04022025_182820_raw.h5ad')
adata

## Validate confidence score and markers expression

In [ ]:
adata_log = adata.copy()
sc.pp.normalize_total(adata_log, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata_log)

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 12))
    sc.pl.umap(adata_log,color=["cellstates_scANVI", "confidence_score"], ncols=1, color_map = 'magma_r', frameon=False, show=False, size = 6)
    plt.savefig(f"figures/manual_validation/immune_scANVI_confidence_{timestamp}.png", bbox_inches="tight")
    plt.show()

## Validate markers expression

In [70]:
def visualize_cell_state_markers(adata, cell_state, markers, timestamp=None):
    """
    Create temporary annotation for specific cell state and visualize with markers.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data matrix
    cell_state : str
        Cell state of interest from cellstates_scANVI column
    markers : list
        List of marker genes to visualize
    timestamp : str, optional
        Timestamp for file naming. If None, current time will be used
    """

    if timestamp is None:
        timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

    # Create temporary column for cell state
    temp_col_name = 'temp_cell_state'
    adata.obs[temp_col_name] = adata.obs['cellstates_scANVI'] == cell_state
    
    # Set up colors in uns
    original_colors = None
    if 'temp_cell_state_colors' in adata.uns:
        original_colors = adata.uns['temp_cell_state_colors'].copy()
    
    # Set up color palette
    adata.uns[f'{temp_col_name}_colors'] = ['#D3D3D3', '#ba0f30']  # Light pink for target, light grey for others
    
    # Calculate plot layout
    n_total_plots = len(markers) + 1  # +1 for cell state plot
    n_cols = 3
    n_rows = (n_total_plots + (n_cols - 1)) // n_cols  # Ceiling division
    
    # Create the plot
    with plt.rc_context():
        sc.set_figure_params(dpi=300, figsize=(15, 5*n_rows))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        
        # Convert axes to 1D array if necessary
        if n_rows == 1:
            axes = np.array([axes])  # Handle single row case
        axes = axes.flatten()
        
        # Plot cell state
        sc.pl.umap(adata, color=[temp_col_name], ax=axes[0], 
                  frameon=False, title=f'{cell_state} cells',
                  show=False, size=3)
        
        # Plot markers
        for i, marker in enumerate(markers):
            if i + 1 >= len(axes):
                break
            if marker in adata.var_names:
                sc.pl.umap(adata, color=marker, ax=axes[i + 1],
                          color_map='magma_r', frameon=False,
                          title=marker, show=False, size=3)
            else:
                print(f"Warning: Marker {marker} not found in the data")
                axes[i + 1].set_title(f"{marker} (not found)")
                axes[i + 1].axis('off')
        
        # Remove empty subplots
        for i in range(len(markers) + 1, len(axes)):
            fig.delaxes(axes[i])
        
        plt.tight_layout()
        # Save figure
        plt.savefig(
            f"figures/manual_validation/{cell_state}_markers_{timestamp}.png", 
            bbox_inches="tight"
        )
        plt.show()
    
    # Clean up: remove temporary column and colors
    del adata.obs[temp_col_name]
    if f'{temp_col_name}_colors' in adata.uns:
        del adata.uns[f'{temp_col_name}_colors']
    
    # Restore original colors if they existed
    if original_colors is not None:
        adata.uns['temp_cell_state_colors'] = original_colors

In [ ]:
adata_log.obs['cellstates_scANVI'].value_counts()

+ Macrophages

In [ ]:
markers = ['CD163', 'C1QB', 'C1QC', 'MERTK', 'CTSC', 'CTSD', 'CD209', 'IGF1', 'SDS']
visualize_cell_state_markers(adata_log, "Macrophages", markers, timestamp)

+ IgA plasma cell

In [ ]:
markers = ['SDC1', 'IGHA1']
visualize_cell_state_markers(adata_log, "IgA plasma cell", markers, timestamp)

+ Dendritic cells

In [ ]:
markers = ['CLEC9A', 'CLEC10A', 'LAMP3', 'CLEC4C', 'JCHAIN','ETV6', 'FLT3', 'PTCRA', 'LILRA4']
visualize_cell_state_markers(adata_log, "Dendritic cells", markers, timestamp)

+ Naive B cells

In [ ]:
markers = ['FCMR', 'VPREB1', 'VPREB3', 'IGLL1', 'STAT1']
visualize_cell_state_markers(adata_log, "Naive B cells", markers, timestamp)

* ILCs

In [ ]:
markers = ['S100A13', 'TLE1','AREG', 'CXCR3', 'CD3D', 'IKZF3', 'GATA3', 'KLRG1', 'HPGDS', 'IL4I1', 'RORC', 'KIT']
visualize_cell_state_markers(adata_log, "ILCs", markers, timestamp)

* Activated CD8 T cells

In [ ]:
markers = ['CD8A']
visualize_cell_state_markers(adata_log, "Activated CD8 T cells", markers, timestamp)

* NK cells

In [ ]:
markers = ['EOMES', 'NKG7', 'PRF1']
visualize_cell_state_markers(adata_log, "NK cells", markers, timestamp)

* IgM plasma cell

In [ ]:
markers = ['CD19', 'CD27','CD38', 'IGHM']
visualize_cell_state_markers(adata_log, "IgM plasma cell", markers, timestamp)

* Activated CD4 T cells

In [ ]:
markers = ['CD4']
visualize_cell_state_markers(adata_log, "Activated CD4 T cells", markers, timestamp)

* Memory B cells

In [ ]:
markers = ['SELL', 'CR2','CD27', 'MS4A1']
visualize_cell_state_markers(adata_log, "Memory B cells", markers, timestamp)

* gamma delta T cells

In [ ]:
markers = ['TRGV2', 'TRDC', 'TRGC1', 'CCL5']
visualize_cell_state_markers(adata_log, "gamma delta T cells", markers, timestamp)

* T helper

In [ ]:
markers = ['CXCR5', 'CXCR3','IFNG', 'RORA', 'IL22','MS4A1', 'CCL5', 'CXCR3', 'TBX21', 'IL7R', 'CCR6', 'ZBTB16']
visualize_cell_state_markers(adata_log, "T helper", markers, timestamp)

* Monocytes

In [ ]:
markers = ['FCN1', 'S100A4', 'S100A6']
visualize_cell_state_markers(adata_log, "Monocytes", markers, timestamp)

* Naive CD4 T cells

In [ ]:
markers = ['CCR7', 'SELL', 'CD4']
visualize_cell_state_markers(adata_log, "Naive CD4 T cells", markers, timestamp)

* Germinal Center B cells

In [ ]:
markers = ['POU2AF1', 'CD40', 'SUGCT']
visualize_cell_state_markers(adata_log, "Germinal Center B cells", markers, timestamp)

* Cycling B cell

In [ ]:
markers = ['MKI67', 'TOP2A', 'CD19']
visualize_cell_state_markers(adata_log, "Cycling B cell", markers, timestamp)

* Early B cells

In [ ]:
markers = ['MKI67', 'MME', 'CD24', 'CD34','SPINK2', 'IL7R', 'ZCCHC7', 'RAG1', 'DNTT', 'IGLL1', 'IGLL5']
visualize_cell_state_markers(adata_log, "Early B cells", markers, timestamp)

* Memory CD8 T cells

In [ ]:
markers = ['CD8A', 'CX3CR1', 'GZMB', 'GNLY', 'FGFBP2','S1PR5', 'FCGR3A','TGFBR3']
visualize_cell_state_markers(adata_log, "Memory CD8 T cells", markers, timestamp)

In [ ]:
adata_log.obs['cellstates_scANVI'].value_counts()

* Naive CD8 T cells

In [ ]:
markers = ['CD8A', 'CCR7', 'SELL']
visualize_cell_state_markers(adata_log, "Naive CD8 T cells", markers, timestamp)

* IgG plasma cell

In [ ]:
markers = ['CD19', 'CD27','CD38', 'IGHG1']
visualize_cell_state_markers(adata_log, "IgG plasma cell", markers, timestamp)

* Treg

In [ ]:
markers = ['CD27', 'CCR7','IKZF4', 'FOXP3', 'CTLA4','TIGIT']
visualize_cell_state_markers(adata_log, "Treg", markers, timestamp)

* Progenitors (ICP)

In [ ]:
markers = ['IL7R', 'KIT','RORC', 'CCR6','NRP1', 'NCR2']
visualize_cell_state_markers(adata_log, "Progenitors", markers, timestamp)

* Mast cells

In [ ]:
markers = ['TPSAB1', 'TPSB2','CPA3']
visualize_cell_state_markers(adata_log, "Mast cells", markers, timestamp)

* NK T cell

In [ ]:
markers = ['NKG7', 'GNLY','CD8A']
visualize_cell_state_markers(adata_log, "NK T cell", markers, timestamp)

* Cycling plasma cell

In [ ]:
markers = ['MKI67', 'TOP2A','CD19']
visualize_cell_state_markers(adata_log, "Cycling plasma cell", markers, timestamp)

* Activated T

In [ ]:
markers = ['CD69', 'HLA-DR', 'CD25', 'ICOS']
visualize_cell_state_markers(adata_log, "Activated T", markers, timestamp)

* MAIT cell

In [ ]:
markers = ['KLRB1', 'SLC4A10','TRAV1-2']
visualize_cell_state_markers(adata_log, "MAIT cell", markers, timestamp)

* Megakaryocyte

In [ ]:
markers = ['CMTM5', 'ITGA2B','PF4']
visualize_cell_state_markers(adata_log, "Megakaryocyte", markers, timestamp)